In [6]:
# ============================================================================
# STAGE 3 - CELL 1: SETUP & DATA LOADING
# ============================================================================

import pandas as pd
import numpy as np
import warnings
warnings.filterwarnings('ignore')

print("\n" + "=" * 80)
print("STAGE 3: BASELINE & FIRST MODEL TRAINING")
print("=" * 80)

print(f"\n📂 CELL 1: SETUP & DATA LOADING")
print("=" * 80)

# Load the cleaned datasets
print(f"\n⏳ Loading datasets...")
train_data = pd.read_csv('kcet_ml_project/data/train_data.csv')
val_data = pd.read_csv('kcet_ml_project/data/val_data.csv')
test_data = pd.read_csv('kcet_ml_project/data/test_data.csv')

print(f"✅ Datasets loaded!")

# Separate features (X) and target (y)
print(f"\n⏳ Separating features and target...")
X_train = train_data.drop('Cutoff_Rank', axis=1)
y_train = train_data['Cutoff_Rank']

X_val = val_data.drop('Cutoff_Rank', axis=1)
y_val = val_data['Cutoff_Rank']

X_test = test_data.drop('Cutoff_Rank', axis=1)
y_test = test_data['Cutoff_Rank']

print(f"✅ Features and targets separated!")

# Data verification
print(f"\n" + "=" * 80)
print(f"📊 DATA VERIFICATION")
print(f"=" * 80)

print(f"\n🎯 Dataset Shapes:")
print(f"   Train: X={X_train.shape}, y={y_train.shape}")
print(f"   Val:   X={X_val.shape}, y={y_val.shape}")
print(f"   Test:  X={X_test.shape}, y={y_test.shape}")

print(f"\n📈 Feature Count:")
print(f"   Total features: {X_train.shape[1]}")
print(f"   All numeric: {X_train.select_dtypes(include=[np.number]).shape[1] == X_train.shape[1]}")

print(f"\n🎯 Target Statistics (Cutoff_Rank):")
print(f"   Train: mean={y_train.mean():,.0f}, std={y_train.std():,.0f}, min={y_train.min():,.0f}, max={y_train.max():,.0f}")
print(f"   Val:   mean={y_val.mean():,.0f}, std={y_val.std():,.0f}, min={y_val.min():,.0f}, max={y_val.max():,.0f}")
print(f"   Test:  mean={y_test.mean():,.0f}, std={y_test.std():,.0f}, min={y_test.min():,.0f}, max={y_test.max():,.0f}")

print(f"\n❌ Missing values:")
print(f"   Train X: {X_train.isnull().sum().sum()}, Train y: {y_train.isnull().sum()}")
print(f"   Val X:   {X_val.isnull().sum().sum()}, Val y:   {y_val.isnull().sum()}")
print(f"   Test X:  {X_test.isnull().sum().sum()}, Test y:  {y_test.isnull().sum()}")

print(f"\n✅ CELL 1 COMPLETE!")
print("=" * 80)



STAGE 3: BASELINE & FIRST MODEL TRAINING

📂 CELL 1: SETUP & DATA LOADING

⏳ Loading datasets...
✅ Datasets loaded!

⏳ Separating features and target...
✅ Features and targets separated!

📊 DATA VERIFICATION

🎯 Dataset Shapes:
   Train: X=(137755, 37), y=(137755,)
   Val:   X=(60681, 37), y=(60681,)
   Test:  X=(71626, 37), y=(71626,)

📈 Feature Count:
   Total features: 37
   All numeric: False

🎯 Target Statistics (Cutoff_Rank):
   Train: mean=69,321, std=45,187, min=90, max=183,210
   Val:   mean=81,605, std=51,665, min=169, max=203,368
   Test:  mean=109,606, std=67,796, min=193, max=274,884

❌ Missing values:
   Train X: 0, Train y: 0
   Val X:   0, Val y:   0
   Test X:  0, Test y:  0

✅ CELL 1 COMPLETE!


In [7]:
# ============================================================================
# STAGE 3 - CELL 1.5: FIND NON-NUMERIC COLUMNS (DIAGNOSTIC)
# ============================================================================

print("\n" + "=" * 80)
print("CELL 1.5: DIAGNOSTIC - FIND NON-NUMERIC COLUMNS")
print("=" * 80)

# Find all non-numeric columns in X_train
non_numeric_cols = X_train.select_dtypes(exclude=[np.number]).columns.tolist()

print(f"\n❌ NON-NUMERIC COLUMNS FOUND: {len(non_numeric_cols)}")

if len(non_numeric_cols) > 0:
    for i, col in enumerate(non_numeric_cols, 1):
        dtype = X_train[col].dtype
        unique = X_train[col].nunique()
        print(f"   {i}. {col:<40} dtype={dtype}, unique={unique}")
else:
    print(f"   None - All columns are numeric!")

# Show all column names for reference
print(f"\n📋 ALL FEATURES IN X_train ({X_train.shape[1]} total):")
for i, col in enumerate(X_train.columns, 1):
    dtype = X_train[col].dtype
    print(f"   {i:2d}. {col:<40} {dtype}")

print("=" * 80)



CELL 1.5: DIAGNOSTIC - FIND NON-NUMERIC COLUMNS

❌ NON-NUMERIC COLUMNS FOUND: 1
   1. year_cohort                              dtype=object, unique=1

📋 ALL FEATURES IN X_train (37 total):
    1. Year                                     int64
    2. Historical_Mean_Primary                  float64
    3. Historical_Mean_Percentile               float64
    4. Historical_Std_Raw                       float64
    5. Cutoff_Rank_raw                          float64
    6. Cutoff_Rank_log1p                        float64
    7. rank_within_category                     float64
    8. Cutoff_Rank_lag1_L1                      float64
    9. Cutoff_Rank_lag2_L1                      float64
   10. Cutoff_Rank_lag3_L1                      float64
   11. Cutoff_Rank_rollmean2_L1                 float64
   12. Cutoff_Rank_rollstd2_L1                  float64
   13. Cutoff_Rank_rollmean3_L1                 float64
   14. Cutoff_Rank_rollstd3_L1                  float64
   15. Cutoff_Rank_lag1_L2  

In [8]:
# ============================================================================
# STAGE 3 - CELL 2: CLEAN FEATURES & LOG TRANSFORM TARGET
# ============================================================================

print("\n" + "=" * 80)
print("CELL 2: CLEAN FEATURES & LOG TRANSFORM TARGET")
print("=" * 80)

# Drop non-numeric columns (year_cohort is metadata, already have Year)
print(f"\n🗑️  DROPPING NON-NUMERIC COLUMNS:")
print(f"   Dropping: year_cohort (metadata, unique=1)")

X_train = X_train.drop(columns=['year_cohort'])
X_val = X_val.drop(columns=['year_cohort'])
X_test = X_test.drop(columns=['year_cohort'])

print(f"   ✅ Dropped!")

# Verify all numeric now
print(f"\n✅ VERIFICATION: All features numeric?")
all_numeric = X_train.select_dtypes(include=[np.number]).shape[1] == X_train.shape[1]
print(f"   {all_numeric} (Features: {X_train.shape[1]})")

# Log transform targets (handles right-skewed distributions)
print(f"\n📊 LOG TRANSFORM TARGET:")
print(f"   Using: log1p(Cutoff_Rank) for training")

y_train_log = np.log1p(y_train)
y_val_log = np.log1p(y_val)
y_test_log = np.log1p(y_test)

print(f"   ✅ Log transformation applied!")

# Statistics
print(f"\n📈 TARGET STATISTICS:")
print(f"\n   Original scale (Cutoff_Rank):")
print(f"      Train: mean={y_train.mean():,.0f}, std={y_train.std():,.0f}")
print(f"      Val:   mean={y_val.mean():,.0f}, std={y_val.std():,.0f}")
print(f"      Test:  mean={y_test.mean():,.0f}, std={y_test.std():,.0f}")

print(f"\n   Log scale (log1p):")
print(f"      Train: mean={y_train_log.mean():.3f}, std={y_train_log.std():.3f}")
print(f"      Val:   mean={y_val_log.mean():.3f}, std={y_val_log.std():.3f}")
print(f"      Test:  mean={y_test_log.mean():.3f}, std={y_test_log.std():.3f}")

print(f"\n✅ CELL 2 COMPLETE!")
print("=" * 80)



CELL 2: CLEAN FEATURES & LOG TRANSFORM TARGET

🗑️  DROPPING NON-NUMERIC COLUMNS:
   Dropping: year_cohort (metadata, unique=1)
   ✅ Dropped!

✅ VERIFICATION: All features numeric?
   True (Features: 36)

📊 LOG TRANSFORM TARGET:
   Using: log1p(Cutoff_Rank) for training
   ✅ Log transformation applied!

📈 TARGET STATISTICS:

   Original scale (Cutoff_Rank):
      Train: mean=69,321, std=45,187
      Val:   mean=81,605, std=51,665
      Test:  mean=109,606, std=67,796

   Log scale (log1p):
      Train: mean=10.850, std=0.901
      Val:   mean=11.028, std=0.879
      Test:  mean=11.335, std=0.861

✅ CELL 2 COMPLETE!


In [9]:
# ============================================================================
# STAGE 3 - CELL 3: BASELINE MODELS (NO ML - SIMPLE BENCHMARKS)
# ============================================================================

from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score

print("\n" + "=" * 80)
print("CELL 3: BASELINE MODELS (ESTABLISH BENCHMARK TO BEAT)")
print("=" * 80)

# Helper function
def calculate_rmse(y_true, y_pred):
    return np.sqrt(mean_squared_error(y_true, y_pred))

# ============================================================================
# BASELINE 1: LAG-1 PERSISTENCE (Simple Persistence)
# ============================================================================

print("\n📊 BASELINE 1: Lag-1 Persistence (Last Year's Rank)")
print("=" * 60)

baseline1_val_pred = X_val['Cutoff_Rank_lag1_L1'].fillna(y_train.median())
baseline1_test_pred = X_test['Cutoff_Rank_lag1_L1'].fillna(y_train.median())

baseline1_val_mae = mean_absolute_error(y_val, baseline1_val_pred)
baseline1_test_mae = mean_absolute_error(y_test, baseline1_test_pred)

baseline1_val_rmse = calculate_rmse(y_val, baseline1_val_pred)
baseline1_test_rmse = calculate_rmse(y_test, baseline1_test_pred)

print(f"   Validation MAE:  {baseline1_val_mae:,.0f}")
print(f"   Validation RMSE: {baseline1_val_rmse:,.0f}")
print(f"   Test MAE:        {baseline1_test_mae:,.0f}")
print(f"   Test RMSE:       {baseline1_test_rmse:,.0f}")

# ============================================================================
# BASELINE 2: MEAN PREDICTION (Global Average)
# ============================================================================

print("\n📊 BASELINE 2: Global Mean (Simple Average)")
print("=" * 60)

global_mean = y_train.mean()

baseline2_val_pred = np.full(len(y_val), global_mean)
baseline2_test_pred = np.full(len(y_test), global_mean)

baseline2_val_mae = mean_absolute_error(y_val, baseline2_val_pred)
baseline2_test_mae = mean_absolute_error(y_test, baseline2_test_pred)

baseline2_val_rmse = calculate_rmse(y_val, baseline2_val_pred)
baseline2_test_rmse = calculate_rmse(y_test, baseline2_test_pred)

print(f"   Global mean: {global_mean:,.0f}")
print(f"   Validation MAE:  {baseline2_val_mae:,.0f}")
print(f"   Validation RMSE: {baseline2_val_rmse:,.0f}")
print(f"   Test MAE:        {baseline2_test_mae:,.0f}")
print(f"   Test RMSE:       {baseline2_test_rmse:,.0f}")

# ============================================================================
# BASELINE 3: MEDIAN PREDICTION
# ============================================================================

print("\n📊 BASELINE 3: Global Median (Robust Average)")
print("=" * 60)

global_median = y_train.median()

baseline3_val_pred = np.full(len(y_val), global_median)
baseline3_test_pred = np.full(len(y_test), global_median)

baseline3_val_mae = mean_absolute_error(y_val, baseline3_val_pred)
baseline3_test_mae = mean_absolute_error(y_test, baseline3_test_pred)

baseline3_val_rmse = calculate_rmse(y_val, baseline3_val_pred)
baseline3_test_rmse = calculate_rmse(y_test, baseline3_test_pred)

print(f"   Global median: {global_median:,.0f}")
print(f"   Validation MAE:  {baseline3_val_mae:,.0f}")
print(f"   Validation RMSE: {baseline3_val_rmse:,.0f}")
print(f"   Test MAE:        {baseline3_test_mae:,.0f}")
print(f"   Test RMSE:       {baseline3_test_rmse:,.0f}")

# ============================================================================
# BASELINE SUMMARY
# ============================================================================

print("\n" + "=" * 80)
print("🎯 BASELINE SUMMARY (VALIDATION SET)")
print("=" * 80)

baselines = [
    ('Lag-1 Persistence', baseline1_val_mae),
    ('Global Mean', baseline2_val_mae),
    ('Global Median', baseline3_val_mae)
]

baselines_sorted = sorted(baselines, key=lambda x: x[1])

for i, (name, mae) in enumerate(baselines_sorted, 1):
    print(f"   {i}. {name:<25} MAE: {mae:,.0f}")

best_baseline_mae = baselines_sorted[0][1]
print(f"\n🏆 BEST BASELINE TO BEAT: {best_baseline_mae:,.0f} MAE")
print(f"   Model must beat this to add value!")

print("\n✅ CELL 3 COMPLETE!")
print("=" * 80)



CELL 3: BASELINE MODELS (ESTABLISH BENCHMARK TO BEAT)

📊 BASELINE 1: Lag-1 Persistence (Last Year's Rank)
   Validation MAE:  26,757
   Validation RMSE: 41,265
   Test MAE:        32,570
   Test RMSE:       49,696

📊 BASELINE 2: Global Mean (Simple Average)
   Global mean: 69,321
   Validation MAE:  41,955
   Validation RMSE: 53,105
   Test MAE:        60,419
   Test RMSE:       78,861

📊 BASELINE 3: Global Median (Robust Average)
   Global median: 61,038
   Validation MAE:  42,986
   Validation RMSE: 55,608
   Test MAE:        63,753
   Test RMSE:       83,397

🎯 BASELINE SUMMARY (VALIDATION SET)
   1. Lag-1 Persistence         MAE: 26,757
   2. Global Mean               MAE: 41,955
   3. Global Median             MAE: 42,986

🏆 BEST BASELINE TO BEAT: 26,757 MAE
   Model must beat this to add value!

✅ CELL 3 COMPLETE!


In [10]:
# ============================================================================
# STAGE 3 - CELL 4: TRAIN LIGHTGBM MODEL
# ============================================================================

import lightgbm as lgb
import time
from lightgbm import early_stopping

print("\n" + "=" * 80)
print("CELL 4: TRAIN LIGHTGBM MODEL")
print("=" * 80)

# Initialize LightGBM Regressor
print(f"\n🔧 INITIALIZING LIGHTGBM...")

lgbm_base = lgb.LGBMRegressor(
    n_estimators=1000,
    learning_rate=0.05,
    num_leaves=31,
    max_depth=7,
    min_child_samples=20,
    subsample=0.8,
    colsample_bytree=0.8,
    random_state=42,
    n_jobs=-1
)

print(f"   ✅ Model initialized with:")
print(f"      Estimators: 1000")
print(f"      Learning rate: 0.05")
print(f"      Max depth: 7")
print(f"      Num leaves: 31")

# Train the model
print(f"\n⏳ TRAINING IN PROGRESS...")
start_time = time.time()

lgbm_base.fit(
    X_train, y_train_log,
    eval_set=[(X_val, y_val_log)],
    eval_metric='mae',
    callbacks=[early_stopping(stopping_rounds=200)]
)

training_time = time.time() - start_time
print(f"✅ Training complete!")
print(f"   Training time: {training_time:.2f} seconds ({training_time/60:.2f} minutes)")

# Generate predictions
print(f"\n🔮 GENERATING PREDICTIONS...")

y_train_pred_log = lgbm_base.predict(X_train)
y_val_pred_log = lgbm_base.predict(X_val)
y_test_pred_log = lgbm_base.predict(X_test)

# Inverse transform (convert log predictions back to original scale)
y_train_pred = np.expm1(y_train_pred_log)
y_val_pred = np.expm1(y_val_pred_log)
y_test_pred = np.expm1(y_test_pred_log)

print(f"   ✅ Predictions generated")
print(f"      Train predictions: {len(y_train_pred):,}")
print(f"      Val predictions: {len(y_val_pred):,}")
print(f"      Test predictions: {len(y_test_pred):,}")

# ============================================================================
# EVALUATE MODEL
# ============================================================================

print(f"\n" + "=" * 80)
print("📊 MODEL EVALUATION")
print("=" * 80)

# Calculate metrics
train_mae = mean_absolute_error(y_train, y_train_pred)
val_mae = mean_absolute_error(y_val, y_val_pred)
test_mae = mean_absolute_error(y_test, y_test_pred)

train_rmse = calculate_rmse(y_train, y_train_pred)
val_rmse = calculate_rmse(y_val, y_val_pred)
test_rmse = calculate_rmse(y_test, y_test_pred)

train_r2 = r2_score(y_train, y_train_pred)
val_r2 = r2_score(y_val, y_val_pred)
test_r2 = r2_score(y_test, y_test_pred)

print(f"\n🎯 MAE (MAIN METRIC):")
print(f"   Train: {train_mae:,.0f}")
print(f"   Val:   {val_mae:,.0f}")
print(f"   Test:  {test_mae:,.0f}")

print(f"\n📏 RMSE:")
print(f"   Train: {train_rmse:,.0f}")
print(f"   Val:   {val_rmse:,.0f}")
print(f"   Test:  {test_rmse:,.0f}")

print(f"\n📈 R² SCORE:")
print(f"   Train: {train_r2:.4f}")
print(f"   Val:   {val_r2:.4f}")
print(f"   Test:  {test_r2:.4f}")

# ============================================================================
# COMPARE WITH BASELINE
# ============================================================================

print(f"\n" + "=" * 80)
print("🔍 COMPARISON WITH BASELINE")
print("=" * 80)

baseline_mae = 26757
improvement_pct = ((baseline_mae - val_mae) / baseline_mae) * 100
improvement_absolute = baseline_mae - val_mae

print(f"\n   Baseline (Lag-1): {baseline_mae:,.0f} MAE")
print(f"   LightGBM Model:   {val_mae:,.0f} MAE")
print(f"\n   ➕ Absolute improvement: {improvement_absolute:+,.0f}")
print(f"   📈 Percentage improvement: {improvement_pct:+.2f}%")

if val_mae < baseline_mae:
    print(f"\n   ✅ SUCCESS! Model beats baseline!")
else:
    print(f"\n   ⚠️ Model underperforms baseline (needs tuning)")

print(f"\n✅ CELL 4 COMPLETE!")
print("=" * 80)



CELL 4: TRAIN LIGHTGBM MODEL

🔧 INITIALIZING LIGHTGBM...
   ✅ Model initialized with:
      Estimators: 1000
      Learning rate: 0.05
      Max depth: 7
      Num leaves: 31

⏳ TRAINING IN PROGRESS...
[LightGBM] [Warning] Accuracy may be bad since you didn't explicitly set num_leaves OR 2^max_depth > num_leaves. (num_leaves=31).
[LightGBM] [Warning] Accuracy may be bad since you didn't explicitly set num_leaves OR 2^max_depth > num_leaves. (num_leaves=31).
[LightGBM] [Warning] Auto-choosing col-wise multi-threading, the overhead of testing was 0.027629 seconds.
You can set `force_col_wise=true` to remove the overhead.
[LightGBM] [Info] Total Bins 8890
[LightGBM] [Info] Number of data points in the train set: 137755, number of used features: 36
[LightGBM] [Warning] Accuracy may be bad since you didn't explicitly set num_leaves OR 2^max_depth > num_leaves. (num_leaves=31).
[LightGBM] [Info] Start training from score 10.850157
Training until validation scores don't improve for 200 round

In [11]:
# ============================================================================
# STAGE 3 - CELL 5: DATA LEAKAGE DETECTION & ANALYSIS
# ============================================================================

print("\n" + "=" * 80)
print("CELL 5: DATA LEAKAGE DETECTION (CRITICAL CHECK)")
print("=" * 80)

# ============================================================================
# 1. CHECK FEATURE IMPORTANCE
# ============================================================================

print(f"\n🔍 FEATURE IMPORTANCE (Top 15):")
print("=" * 60)

feature_importance = pd.DataFrame({
    'feature': X_train.columns,
    'importance': lgbm_base.feature_importances_
}).sort_values('importance', ascending=False)

print(feature_importance.head(15).to_string(index=False))

print(f"\n⚠️ ANALYSIS:")
print(f"   If lag/historical features dominate → LIKELY DATA LEAKAGE!")
print(f"   Expected: Target encodings + aggregate features important")
print(f"   Unexpected: Lag features having 80%+ importance → LEAKAGE!")

# ============================================================================
# 2. CHECK LAG FEATURE CORRELATION WITH PREDICTIONS
# ============================================================================

print(f"\n🔍 LAG FEATURE ANALYSIS:")
print("=" * 60)

lag_features = [col for col in X_val.columns if 'lag' in col.lower() or 'roll' in col.lower()]
print(f"   Total lag/roll features: {len(lag_features)}")
print(f"   Features: {lag_features}")

# Calculate correlation between lag features and predictions
lag_importance_total = feature_importance[feature_importance['feature'].isin(lag_features)]['importance'].sum()
total_importance = feature_importance['importance'].sum()
lag_importance_pct = (lag_importance_total / total_importance) * 100

print(f"\n   Lag/Roll feature importance: {lag_importance_pct:.1f}% of total")

if lag_importance_pct > 70:
    print(f"\n   🚨 RED FLAG: Lag features are TOO important!")
    print(f"      This suggests data leakage!")
else:
    print(f"\n   ✅ Normal: Lag features are secondary")

# ============================================================================
# 3. RESIDUAL ANALYSIS
# ============================================================================

print(f"\n🔍 RESIDUAL ANALYSIS:")
print("=" * 60)

residuals_val = y_val - y_val_pred
residuals_test = y_test - y_test_pred

print(f"   Validation residuals:")
print(f"      Mean: {residuals_val.mean():,.0f} (should be ~0)")
print(f"      Std:  {residuals_val.std():,.0f}")
print(f"      Max:  {residuals_val.max():,.0f}")
print(f"      Min:  {residuals_val.min():,.0f}")

print(f"\n   Test residuals:")
print(f"      Mean: {residuals_test.mean():,.0f} (should be ~0)")
print(f"      Std:  {residuals_test.std():,.0f}")
print(f"      Max:  {residuals_test.max():,.0f}")
print(f"      Min:  {residuals_test.min():,.0f}")

print(f"\n   If Val residuals are MUCH smaller than Test → LEAKAGE!")
print(f"   Current gap (RMSE): Val={val_rmse:,.0f} vs Test={test_rmse:,.0f}")

gap_ratio = test_rmse / val_rmse
print(f"   Gap ratio (Test/Val): {gap_ratio:.2f}x")

if gap_ratio > 6:
    print(f"   🚨 RED FLAG: Large gap indicates LEAKAGE in validation set!")
else:
    print(f"   ✅ Normal: Reasonable generalization")

# ============================================================================
# 4. TIME-SERIES LEAKAGE CHECK
# ============================================================================

print(f"\n🔍 TIME-SERIES LEAKAGE CHECK:")
print("=" * 60)

print(f"   Train years: {sorted(X_train['Year'].unique())}")
print(f"   Val years:   {sorted(X_val['Year'].unique())}")
print(f"   Test years:  {sorted(X_test['Year'].unique())}")

print(f"\n   ✅ Good: Train < Val < Test (proper temporal split)")

print(f"\n✅ CELL 5 COMPLETE!")
print("=" * 80)



CELL 5: DATA LEAKAGE DETECTION (CRITICAL CHECK)

🔍 FEATURE IMPORTANCE (Top 15):
                            feature  importance
                    Cutoff_Rank_raw        2730
               rank_within_category         834
                     category_ratio         475
                  Cutoff_Rank_log1p         408
            Cutoff_Rank_rollstd2_L1         370
         Branch_Category_target_enc         308
            Cutoff_Rank_rollstd2_L2         298
 branch_global_avg_cutoff_last_year         274
        College_Category_target_enc         263
            Cutoff_Rank_rollstd3_L1         242
                Cutoff_Rank_lag3_L1         236
            Cutoff_Rank_rollstd2_L3         223
                  branch_volatility         207
college_cutoff_std_allbranches_past         200
            Cutoff_Rank_rollstd3_L2         197

⚠️ ANALYSIS:
   If lag/historical features dominate → LIKELY DATA LEAKAGE!
   Expected: Target encodings + aggregate features important
   Unexpected: